# Task 15: Distributed Data Parallel (DDP)
Objective: Master large-scale deep learning deployment strategies by orchestrating
distributed training runs synced across isolated hardware nodes.


In [1]:
import torch
import torch.nn as nn
import torch.distributed as dist

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

PyTorch: 2.11.0+cpu
CUDA available: False
GPU count: 0


## 1. Simple Model

In [2]:
class SimpleModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(10, 32),
            nn.ReLU(),
            nn.Linear(32, 2)
        )

    def forward(self, x):
        return self.net(x)

## 2. DDP Training Function

In [3]:
def train_ddp(rank, world_size):

    # Initialize communication between processes.
    dist.init_process_group(
        backend="nccl",
        rank=rank,
        world_size=world_size
    )

    torch.cuda.set_device(rank)

    model = SimpleModel().cuda(rank)

    # Wrap model with DistributedDataParallel.
    model = nn.parallel.DistributedDataParallel(
        model,
        device_ids=[rank]
    )

    # Simple training data.
    X = torch.randn(
        1000, 10,
        device=rank
    )

    y = torch.randint(
        0, 2,
        (1000,),
        device=rank
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001
    )

    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(5):

        optimizer.zero_grad()

        output = model(X)

        loss = loss_fn(
            output,
            y
        )

        loss.backward()

        # DDP automatically synchronizes gradients.
        optimizer.step()

        if rank == 0:
            print(
                f"Epoch {epoch + 1} "
                f"Loss: {loss.item():.4f}"
            )

    dist.destroy_process_group()

## 3. Launch Multiple GPU Processes



In [4]:
# Example launch command:
#
# torchrun --nproc_per_node=2 train.py
#
# Inside train.py:
#
# import os
# rank = int(os.environ["LOCAL_RANK"])
# world_size = int(os.environ["WORLD_SIZE"])
# train_ddp(rank, world_size)

print("DDP training function is ready.")

DDP training function is ready.
